# 🔍 Distributed Tracing

**Track requests across services**

## 📋 Overview

**What you'll learn:**
- OpenTelemetry
- Trace context propagation
- Spans and traces
- Jaeger integration

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

## 🔍 OpenTelemetry Setup

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.jaeger.thrift import JaegerExporter

# Setup tracer
trace.set_tracer_provider(TracerProvider())
tracer = trace.get_tracer(__name__)

# Configure Jaeger exporter
jaeger_exporter = JaegerExporter(
    agent_host_name='localhost',
    agent_port=6831,
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(jaeger_exporter)
)
```

## 📊 Tracing LLM Requests

```python
def chat_with_tracing(message: str):
    # Create span for the entire request
    with tracer.start_as_current_span('chat_request') as span:
        span.set_attribute('user_message', message)
        
        # Check cache
        with tracer.start_as_current_span('cache_lookup'):
            cached = cache.get(message)
            if cached:
                span.set_attribute('cache_hit', True)
                return cached
        
        # Call LLM
        with tracer.start_as_current_span('llm_call') as llm_span:
            llm_span.set_attribute('model', 'gpt-4')
            
            start = time.time()
            response = client.chat.completions.create(
                model='gpt-4',
                messages=[{'role': 'user', 'content': message}]
            )
            
            llm_span.set_attribute('latency_ms', (time.time() - start) * 1000)
            llm_span.set_attribute('tokens', response.usage.total_tokens)
        
        # Cache result
        with tracer.start_as_current_span('cache_store'):
            cache.set(message, response)
        
        return response
```

**Trace visualization shows:**
```
chat_request (1.5s)
├── cache_lookup (5ms) ❌ miss
├── llm_call (1.2s)
│   └── tokens: 150
└── cache_store (2ms)
```

## ✅ Summary

**Tracing benefits:**
- See request flow across services
- Identify bottlenecks
- Debug complex issues
- Monitor service dependencies

**Key concepts:**
- **Trace**: End-to-end request journey
- **Span**: Single operation
- **Context**: Links spans together

### Next: `11_observability/04_monitoring.ipynb`